# Structured LLM Output: Filling Schemas with a Model [Step 3 - Structured LLM output]

> **MLCourse - Agentic AI - Output Parsers and Pydantic**

> Stage in the capstone: capstone answers are parsed into a strict AnswerWithSources
> schema so citations cannot hallucinate.

### What you'll learn
- defining the `AnswerWithSources` contract with descriptions and constraints
- Route A: `llm.with_structured_output(Model)` - the native, preferred path that returns model instances
- Route B: `PydanticOutputParser` fallback - format instructions injected into the prompt, `OutputParserException` handled
- Route C: parsing a hand-written JSON payload through the SAME parser with zero API keys

Notebook 01 of this module built contracts in pure pydantic. Now we make real chat
models fill them - and keep every live call behind an explicit guard.

### Standard first cell for every MLCourse notebook: imports, inline plotting,


In [ ]:
# and automatic discovery of the track-level .env file.
import os                                   # read environment variables such as GROQ_API_KEY
from pathlib import Path                    # walk up the folder tree hunting for .env

try:                                        # Jupyter kernels define get_ipython();
    get_ipython().run_line_magic("matplotlib", "inline")  # render plots inside the notebook
except NameError:                           # plain python runs have no IPython,
    pass                                    # so skip the magic silently

from dotenv import load_dotenv              # loads KEY=VALUE lines into os.environ


def find_track_env(start: Path) -> "Path | None":
    """Climb from *start* upward until 03_agentic_ai/.env appears."""
    for folder in (start, *start.parents):             # current dir, then every parent
        candidate = folder / "03_agentic_ai" / ".env"  # track secrets live at this spot
        if candidate.is_file():                        # hit: stop climbing immediately
            return candidate
    return None                                        # miss everywhere: caller decides


_env_path = find_track_env(Path.cwd())     # search from wherever the kernel started
if _env_path is not None:                  # found the track root?
    load_dotenv(_env_path)                 # push GROQ_API_KEY etc. into os.environ
    print("[setup] loaded env:", _env_path)
else:
    print("[setup] no 03_agentic_ai/.env found - live demos will be skipped")


### 1. The contract: AnswerWithSources

All three routes below target the SAME schema. Field descriptions matter twice:
humans read them during code review, and several routes ship them verbatim to the
model as instruction text. Vague descriptions produce vague fills.

> **Pro tip:** treat `description=` strings as prompt copy under version control -
> tightening "sources" to say "real URLs only" measurably reduces fabricated links.

In [2]:
import json                                 # pretty-printing the generated JSON Schema
from pydantic import BaseModel, Field       # contract toolkit from the previous notebook


class AnswerWithSources(BaseModel):
    """The capstone answer shape: text, receipts, and self-rated certainty."""

    answer: str = Field(description="Direct answer to the question, one to three sentences")
    sources: list[str] = Field(description="URLs or titles backing the answer; real references only")
    confidence: float = Field(ge=0.0, le=1.0, description="Certainty between 0 and 1")


schema_doc = AnswerWithSources.model_json_schema()      # what parsers will tell the model
print(json.dumps(schema_doc, indent=2)[:600])           # peek at the machine-generated contract

{
  "description": "The capstone answer shape: text, receipts, and self-rated certainty.",
  "properties": {
    "answer": {
      "description": "Direct answer to the question, one to three sentences",
      "title": "Answer",
      "type": "string"
    },
    "sources": {
      "description": "URLs or titles backing the answer; real references only",
      "items": {
        "type": "string"
      },
      "title": "Sources",
      "type": "array"
    },
    "confidence": {
      "description": "Certainty between 0 and 1",
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidenc


### 2. Route A - native structured output (preferred)

`llm.with_structured_output(AnswerWithSources)` asks the PROVIDER for its JSON or
tool-calling mode and parses server-side against your schema. The chain then returns
a real `AnswerWithSources` instance - no prompt surgery, no brittle text scraping,
fewer failure modes. Use this whenever the backend supports it.

> **Common pitfall:** once you use `with_structured_output`, do NOT also paste
> format instructions into the prompt. Two competing contracts confuse the model;
> the schema itself is now the single source of truth.

In [3]:
if not os.getenv("GROQ_API_KEY"):           # mandatory guard before any provider call
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to run this live call.")
else:
    from langchain_groq import ChatGroq     # official Groq integration package
    from langchain_core.prompts import ChatPromptTemplate

    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)  # deterministic fills
    strict_llm = llm.with_structured_output(AnswerWithSources)      # replies BECOME instances

    route_a_prompt = ChatPromptTemplate.from_messages([
        ("system", "You answer factual questions and always cite where information came from."),
        ("human", "{question}"),
    ])
    route_a = route_a_prompt | strict_llm   # note: NO parser at the end - built in already
    result = route_a.invoke({"question": "Why is the sky blue? Keep it brief."})

    print(type(result).__name__)            # AnswerWithSources - already validated
    print("answer:", result.answer[:80])
    print("sources:", result.sources)
    print("confidence:", result.confidence)

AnswerWithSources
answer: The sky appears blue because sunlight is scattered by air molecules; shorter blu
sources: ['https://www.nasa.gov/audience/forstudents/5-8/features/nasa-knows/what-is-scattering-58.html', 'https://www.scientificamerican.com/article/why-is-the-sky-blue/']
confidence: 0.95


### 3. Route B - prompt-and-parse fallback

Models without a native structured mode still obey FORMAT RULES printed inside the
prompt. `PydanticOutputParser.get_format_instructions()` renders our schema as such
a rule sheet; `.partial(...)` embeds it so callers never see it. The parser then
converts raw text into the instance - and raises `OutputParserException` when the
model drifts off-contract (prose around the JSON, trailing commas, wrong types).

> **Common pitfall:** creating the parser but never injecting its instructions. The
> parser validates; it does NOT teach. If `{format_instructions}` never reaches the
> template, you are betting the model guesses the shape correctly.

In [4]:
from langchain_core.prompts import ChatPromptTemplate   # used below

try:                                        # import location stabilized in recent cores
    from langchain_core.output_parsers import PydanticOutputParser
except ImportError:                         # older LangChain stacks expose it via langchain
    from langchain.output_parsers import PydanticOutputParser

from langchain_core.exceptions import OutputParserException   # raised when text breaks contract


parser_b = PydanticOutputParser(pydantic_object=AnswerWithSources)   # bind parser to OUR schema

format_instructions = parser_b.get_format_instructions()   # THE rule sheet for the prompt
print(format_instructions[:500])            # show the injected text verbatim (first 500 chars)
print("... [full length:", len(format_instructions), "characters]")

route_b_prompt = ChatPromptTemplate.from_messages([
    ("system", "Follow the output format EXACTLY.\n{format_instructions}"),
    ("human", "{question}"),
]).partial(format_instructions=format_instructions)   # pre-filled at build time


def make_route_b_model():
    """Prefer Groq; fall back to local Ollama; report and give up otherwise."""
    if os.getenv("GROQ_API_KEY"):
        from langchain_groq import ChatGroq
        return ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    try:                                    # local llama3.2: free, needs Ollama installed+running
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.2")
    except Exception:                       # package missing or server down
        print("[demo skipped] ollama pull llama3.2")
        return None


route_b_model = make_route_b_model()
if route_b_model is not None:
    route_b = route_b_prompt | route_b_model | parser_b   # parser rides at the END of the pipe
    try:
        parsed = route_b.invoke(
            {"question": "Name one planet with rings, and give a source."}
        )
    except OutputParserException as exc:    # the model drifted off-contract
        print("[teaching moment] parse failed:", str(exc)[:200])
        print("Fixes: retry the call, tighten the system rule, or move to Route A.")
    except Exception as exc:                # Ollama server down / no network / bad key
        print("[demo skipped] provider unreachable:", str(exc)[:90])
        print("Start Ollama (ollama pull llama3.2) or set GROQ_API_KEY, then rerun.")
    else:
        print(type(parsed).__name__, "->", parsed.answer[:60])   # validated instance

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "The capstone answer shape: text,
... [full length: 1014 characters]


AnswerWithSources -> Saturn


### 4. Offline replay: the same parser, zero providers

Parsing is ordinary code - no network required. Feed the parser a HAND-WRITTEN JSON
payload to watch success, then a prose twin to meet the exception deterministically.
This keeps the lesson runnable in classrooms and CI without any API key.

### Pretend a model returned exactly this payload (plain JSON, nothing else):


In [ ]:
good_json = """
{
  "answer": "Saturn has the most visible ring system of the planets.",
  "sources": ["nasa.gov/saturn/rings"],
  "confidence": 0.91
}
"""
obj = parser_b.parse(good_json)             # same entry point the Route B chain used
print(obj.answer)                           # typed access; constraints already enforced
print("confidence sane:", 0.0 <= obj.confidence <= 1.0)

bad_json = "Sure! Saturn has rings. Confidence: pretty high."   # prose instead of JSON
try:
    parser_b.parse(bad_json)                # deliberate off-contract input
except OutputParserException as exc:        # deterministic teaching moment, no keys needed
    print("[expected failure]", str(exc)[:160])


### Summary

- Route A (`with_structured_output`) is the shortest reliable path: schema in,
  validated instance out - and no extra instructions in the prompt.
- Route B (`PydanticOutputParser`) travels anywhere: embed `get_format_instructions()`
  in the prompt, pipe the parser LAST, expect `OutputParserException` sometimes.
- The offline replay proves the point: the parser is just strict code, so the same
  schema + hand-written JSON teaches with zero providers.
- In the capstone, `AnswerWithSources` is the gate: answers that fail validation
  never reach the user, which is exactly how hallucinated citations get stopped.

Module 04 next: these parsers become ordinary stages inside LCEL chains.